# CRUD, formulaires, gestions d'utilisateurs


In [70]:
# on restaure notre base de données en remplaçant la BDD par son backup 
!cp ./richelieu.db.bak ./richelieu.db

On sait maintenant:
- créer une appli Flask
- la connecter à une base de données via SQLAlchemy  
- modéliser une BDD via l'ORM SQLAlchemy
- parcourir les objets de l'ORM, et leurs relations
- faire des requêtes `SELECT` avec SQLAlchemy
- créer des templates Jinja

**En bref**: tout pour faire un site "catalogue", qui affiche des données sans pouvoir .

Aujourd'hui, on va voir comment faire **toutes les opérations CRUD** sur notre BDD directement depuis le site. On va donc apprendre à:
- **faire des requêtes `CREATE`, `UPDATE` et `DELETE`** avec SQLAlchemy (on sait déjà faire le `READ`, on saura donc faire toutes les opérations CRUD) 
- **créer des formulaires HTML** pour ajouter/modifier/supprimer des données depuis le site

---

# Apparté bonnes pratiques

On code maintenant ensemble depuis 8h et on va commencer à voir des fonctions un peu plus longues.

Rappelez vous que **vous codez pour le long terme**: votre code devra être utilisé par vous et par d'autres, pendant plusieurs années. Retenez que c'est **plus dur de lire du code que d'en écrire** (il faut s'adapter à une autre logique, traduire le code en langage humain...), et c'est **beaucoup plus dur de lire du code de quelqu'un d'autre** (il faut s'adapter à la logique de quelqu'un d'autre). **Pour coder, il faut avoir beaucoup d'informations en tête** (noms de variables, ce qu'elles contiennent, ce qu'on veut en faire, comment...). 

Votre but, c'est d'être **le plus descriptif pour réduire le volume d'informations que vous devez retenir** dans votre "RAM mentale".

**Voici donc quelques bonnes pratiques à adopter**:
- **commentez votre code**: chaque fonction doit avec une docstring
- **une fonction = une opération**: une fonction doit faire une chose, et le faire bien. Plus une fonction est longue, 
    - plus elle est complexe 
    - dure à comprendre
    - plus il y a un risque d'erreur
- **faites des fonctions courtes**: une fonction doit faire au grand maximum la hauteur de votre écran (format paysage)  
- **soyez explicites** dans nos noms de variables, de fonctions et de classes:
    - une variable appelée `x` ou une fonction `f`, ça ne dit pas grand chose sur ce que contient la variable, ou ce que la fonciton fait
    - **votre objectif**:
        - vos variables décrivent leur contenu
        - vos noms de fonction décrivent l'opération faite par la fonction 
        - écrire du code synthétique, c'est moins important qu'écrire du code compréhensible !
- **typez vos fonctions**: peut-être le plus important. **Typer, c'est expliciter ce que contient une variable, et ce que fait une fonction**. C'est **très douloureux** de devoir mettre des prints partout pour comprendre ce que contiennent des variables, et les type hints permettent d'alléger beaucoup ça.

En bref, 

```py
# cette fonction
def f(a, b):
    if len(a) > b:
        return f"{a[:b]}[...]"
    return a

# est beaucoup moins claire que:
def raccourcir(string, max_len):
    if len(string) > max_len:
        return f"{string[:max_len]}[...]"
    return string

# qui est moins claire que:
def raccourcir_avec_types(string: str, max_len: int) -> str:
    if len(string) > max_len:
        return f"{string[:max_len]}[...]"
    return string
```



---

# Sécurisation de l'application: la `SECRET_KEY`

En interne, Flask doit confirmer l'authenticité de plein de choses. Dans ce qui va suivre, Flask a par exemple besoin de confirmer que les données d'un formulaire reçues sont authentiques. Pour cela, Flask a besoin que l'on définisse une `SECRET_KEY`.

**On définit `SECRET_KEY`** dans `app/utils/constants.py`:
```py
from warnings import warn

# la clé top secrète
secret_key_default = "Une clé secrète"
SECRET_KEY = "Une clé secrète"

if SECRET_KEY == secret_key_default:
    warn(f"Changez votre clé secrète avant de passer en production ! Clé secrète actuelle: {SECRET_KEY}")
```

Et dans `app/app.py`, **on ajoute `SECRET_KEY` à la configuration de notre appli** Flask:

```py
app = Flask(
    APP_NAME,
    template_folder=DIR_TEMPLATES, 
    static_folder=DIR_STATICS
)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{PATH_DB}"
app.config["SECRET_KEY"] = SECRET_KEY
db = SQLAlchemy(app)
```

**À noter**: une `SECRET_KEY` doit toujours être confidentielle. C'est pourquoi dans `constants.py`, on voit un `warn()`. **Avant de mettre une appli en production, il faut toujours définir une clé secrète *random*** via un algorithme solide.

> **Lancez l'appli `apps/s5/secret_key` et regardez le warning qui s'affiche (le reste de l'appli n'a pas changé)**
> ```py
> python apps/s5/secret_key/main.py
> ```


---

# Première requête `CREATE`: créer un nouvel objet `Iconography`

On va maintenant apprendre à **créer des nouvelles ressources iconographiques**.


In [71]:
# NOTE: pour le notebook, je combine dans un seul bloc la définition
# de l'appli et tous les modèles de la base de données
# on a donc déjà vu tout le code en dessous.

from typing import List, Optional, Dict
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from werkzeug.security import generate_password_hash
from sqlalchemy import ForeignKey, JSON
from sqlalchemy.orm import Mapped, mapped_column, relationship

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)


class IconographyPlace(db.Model):
    __tablename__ = "iconography_place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_place: Mapped[int] = mapped_column(ForeignKey("place.id"))


class IconographyTheme(db.Model):
    __tablename__ = "iconography_theme"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_theme: Mapped[int] = mapped_column(ForeignKey("theme.id"))


class Author(db.Model):
    __tablename__ = "author"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    author_name: Mapped[str] = mapped_column(unique=True)

    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="author", 
    )


class Theme(db.Model):
    __tablename__ = "theme"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    theme_name: Mapped[str] = mapped_column(unique=True)
    richelieu_url: Mapped[str] = mapped_column(unique=True)

    iconography: Mapped[List["Iconography"]] = relationship(
        secondary=IconographyTheme.__table__,
        back_populates="theme"
    )


class Place(db.Model):
    __tablename__ = "place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    address: Mapped[Optional[str]]
    richelieu_url: Mapped[str] = mapped_column(unique=True)
    loc: Mapped[Dict] = mapped_column(JSON)
    plot: Mapped[Dict] = mapped_column(JSON)
    date_lower: Mapped[int]
    date_upper: Mapped[int]

    iconography: Mapped[List["Iconography"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="place"
    )


class Iconography(db.Model):
    __tablename__ = "iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    id_author: Mapped[Optional[int]] = mapped_column(ForeignKey("author.id"))
    
    author: Mapped[Optional["Author"]] = relationship(back_populates="iconography")
    theme: Mapped[List["Theme"]] = relationship(
        secondary=IconographyTheme.__table__,
        back_populates="iconography"
    )
    place: Mapped[List["Place"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="iconography"
    )


/home/paul/Documents/cours/tnah_devapp/richelieu.db


## 1. Créer un objet `Iconography` côté Python

Pour créer un `Iconography`, rien de plus facile. Nos modèles de base de données sont des classes, donc `Iconography` est une classe et **chaque ressource iconographique est une instance de cette classe**.

On peut trouver des plusieurs ressources iconographiques à ajouter dans [`./data/iconography_sample.md`](./data/iconography_sample.md) 

In [72]:
new_icono = Iconography(
    title="Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]",
    iiif_manifest_url="https://gallica.bnf.fr/iiif/ark:/12148/btv1b530124015/manifest.json",
    iiif_image_url="https://gallica.bnf.fr/iiif/ark:/12148/btv1b530124015/f1/full/1000/0/native.jpg",
    source_url="https://gallica.bnf.fr/ark:/12148/btv1b530124015",
    richelieu_url="https://quartier-richelieu.inha.fr/iconographie/qr1703e89942a0c4fd08052e91956f67719",
    institution="Bibliothèque nationale de France",
    date_lower=1807,
    date_upper=1808,
) 
print(new_icono)
print("title:", new_icono.title)
print("date_lower:", new_icono.date_lower)
print("id_author:", new_icono.id_author)
print("author:", new_icono.author)


<Iconography (transient 124497173589088)>
title: Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]
date_lower: 1807
id_author: None
author: None


**Explication de code**: 
- `Iconography()` permet de créer une instance de classe
- les arguments qu'on passe à `Iconography()` sont les valeurs qu'on veut donner à cette instance (équivalent SQL: les valeurs que l'on donne à chaque colonne)

## 2. Créer les relations

Pour le moment, on a créé un `new_iconography` contenant tout ce qui a trait à la table `Iconography`. Mais, comme on l'a vu dans notre modèle, `Iconography` a aussi des relations vers 3 autres modèles: `Author`, `Place` et `Theme`. 

On va donc maintenant ajouter ces relations à `new_iconography`. Pour ça, il suffit de **séléctionner les objets `Author`, `Place` et `Theme` pertinents et les utiliser pour définir les `relationships` de `new_iconography`**.

In [73]:
with app.app_context():
    author = db.session.get(Author, 235)
    if author:
        print("auteur.ice sélectionné.e:", author)
        new_icono.author = author
    theme = db.session.get(Theme, 13)
    if theme:
        print("thème sélectionné:", author)
        # pourquoi utilise-t'on `append` au lieu de `=` ici ?
        new_icono.theme.append(theme)
    place = db.session.get(Place, 7)
    if place:
        print("lieu sélectionné:", author)
        new_icono.place.append(place)

print("new_icono.author:", new_icono.author)
print("new_icono.place:", new_icono.place)
print("new_icono.theme:", new_icono.theme)
print("adresse:", new_icono.place[0].address)


auteur.ice sélectionné.e: <Author 235>
thème sélectionné: <Author 235>
lieu sélectionné: <Author 235>
new_icono.author: <Author 235>
new_icono.place: [<Place 7>]
new_icono.theme: [<Theme 13>]
adresse: 6, rue Vivienne


/tmp/ipykernel_30224/2625063274.py:6: SAWarning: Object of type <Iconography> not in session, add operation along 'Author.iconography' will not proceed (This warning originated from the Session 'autoflush' process, which was invoked automatically in response to a user-initiated operation. Consider using ``no_autoflush`` context manager if this warning happened while initializing objects.)
  theme = db.session.get(Theme, 13)
/tmp/ipykernel_30224/2625063274.py:11: SAWarning: Object of type <Iconography> not in session, add operation along 'Theme.iconography' won't proceed (This warning originated from the Session 'autoflush' process, which was invoked automatically in response to a user-initiated operation. Consider using ``no_autoflush`` context manager if this warning happened while initializing objects.)
  place = db.session.get(Place, 7)


**Explication de code**:
- on entre dans un `app_context` pour pouvoir faire des requêtes SQL en dehors d'une route
- **on séléctionne les objets liés** (les thèmes, auteurices, lieux) par leur ID
- **on les ajoute à `new_icono`** (`new_icono.author`, `new_icono.place`, `new_icono.theme`)
- **pour les relations many-to-many** (`Iconography.place`, `Iconography.theme`): **les relations sont des listes d'objets** (`Iconography.place` stocke une liste d'objets `Place`). On ajoute donc les objets séléctionnés avec `get` aux listes `new_icono.place` et `new_icono.theme`, plutôt que d'utiliser `=`

Logiquement, si l'objet que l'on veut ajouter en base n'a pas de relations, alors on peut sauter cette 2e étape et passer à la 3e.

## 3. Sauvegarder `new_icono` en base

Pour le moment, `new_icono` n'a pas été sauvegardé. En fait, aucune interaction avec la base de données n'a encore eu lieu.

**Il faut faire un `commit` dans la base de données** pour enregistrer `new_icono`:

In [74]:
# pour rappel, on doit utiliser app.app_context dans les notebooks parce qu'on est hors d'une application Flask. 
with app.app_context():
    try:
        db.session.add(new_icono)
        db.session.commit()
        # on remarque maintenant que `new_icono` a un ID. 
        new_icono_id = new_icono.id

        print(new_icono)
        print(new_icono_id)
        print(new_icono.id_author)
        print(new_icono.author.author_name)
    
    except Exception as e:
        print("Erreur:", e)


<Iconography 1312>
1312
235
Guillet. Lithographe


**Explication de code**:
- `db.session.add()`: on ajoute `new_icono` à la session pour qu'il soit sauvegardé
- `db.session.commit()` sauvegarde tous les changements réalisés pendant une session (create/update/delete).

Le `commit`, si vous vous rappelez, est à la base du SQL: toutes les interactions avec une base de données ont lieu dans une **transaction**, qui est composée de une ou plusieurs requêtes SQL et qui se termine par un **commit**, qui sauvegarde tous les changements de base de données qui ont eu lieu pendant la transaction.

> Pour voir le résultat, on lance `apps/s4/sqlalchemy_many_to_many` et **on va voir `http://localhost:5000/iconographie/$new_icono_id`** (en remplaçant `$new_icono_id` par l'ID de la nouvelle ressource iconographique)
> ```py
> python apps/s4/sqlalchemy_many_to_many/main.py
> ```

## Résumé et remarques

**En résumé**, les étapes à suivre:
- **on crée une nouvelle instance** de la classe `Iconography` (ou de toute autre classe `db.Model`). On renseigne les valeurs de chaque propriété de cette classe.
- **si besoin, on définit les relations** en récupérant les objets pertinents et en les ajoutants aux champs `relationship` (`Iconography.author`...)
- **on ajoute le nouvel objet à la session SQLAlchemy et on le `commit`**.

**À noter**: 
- SQLAlchemy **ne valide les données que juste avant de faire un `commit`**. Avant, il n'y aura pas d'erreur si on met des mauvaises valeurs pour les colonnes d'un modèle (i.e., si on ajoute du texte dans un champ qui attend des nombres)
- **SQLite vérifie aussi nos données avant toute insertion**, puisqu'il dispose de son propre système de validation.

**Dans notre appli, il est donc très important de**:
- ajouter de la **validation de données** avant un insert pour être sûr que ce que l'on insère correspond aux contraintes de nos modèles.
- utiliser des **`try...except` autour de notre `commit()`** pour pouvoir gérer les erreurs d'insertion.




---

# Créer une `Iconography` depuis l'application

Dans `Iconography`, on a maintenant une méthode `Iconography.create()` qui permet de créer des nouvelles ressources iconographiques. **Pour compléter, il nous manque**:
- **une route** pour créer des ressources icono
- **une page HTML** avec un formulaire qui permet de créer celles-ci.

On va donc voir:
- comment créer des formulaires et les valider
- comment créer une route pour pouvoir créer des `Iconography` à partir de données fournies par les utilisateur.ice.s

## Organisation du code

On va travailler sur les fichiers suivants de notre application (`./apps/s5/create_iconography/`):

```txt
└── app
    ├── models
    │   ├── forms.py    # nouveau fichier qui contient tout le code python pour créer des formulaires
    │   └── data.py     # on ajoute `Iconography.create()`
    ├── routes
    │   └── generic.py  # on ajoute ici une route pour créer une nouvelle ressource 
    └── templates
        ├── pages
        │   └── icono_create.html  # template HTML pour créer une ressource icono.
        └── base.html   # on ajoute à notre base de quoi afficher les messages d'erreur
```

## 1. Créer une ressource iconographique depuis `Iconography`: `Iconography.create()`

On a vu comment créer un objet `Iconography` et le sauvegarder en base. On va maintenant **grouper tout ee code vu au dessus dans une seule fonction**

**`Iconography.create()`** est une fonction qui gère la création d'objets iconographiques. Elle prend en paramètres toutes les valeurs que l'on veut donner à notre ressource iconographique, crée le nouvel objet `Iconography` et le sauvegarde en base

Regardons bien le code ci-dessous:

```py
class Iconography(db.Model):
    __tablename__ = "iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    id_author: Mapped[Optional[int]] = mapped_column(ForeignKey("author.id"))
    
    author: Mapped[Optional["Author"]] = relationship(back_populates="iconography")
    theme: Mapped[List["Theme"]] = relationship(
        secondary=IconographyTheme.__table__,
        back_populates="iconography"
    )
    place: Mapped[List["Place"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="iconography"
    )

    @staticmethod 
    def create(
        title: str,
        iiif_manifest_url: str,
        iiif_image_url: str,
        source_url: Optional[str],
        richelieu_url: str,
        date_lower: Optional[int],
        date_upper: Optional[int],
        institution: str,
        id_author: Optional[int],
        id_place: Optional[int],
        id_theme: Optional[int],
    ) -> Tuple[bool, Union["Iconography",str]]:
        """
        créer un nouvel objet icono.
        :returns: (bool, icono|str).
            - bool indique le succès de l'insertion (si True, l'insertion a réussi)
            - si l'insertion a réussi, on retourne l'objet inséré, sinon on retourne un message d'erreur.
        """

        # 1. on crée un nouvel objet Iconography
        new_icono = Iconography(
            title=title,
            iiif_manifest_url=iiif_manifest_url,
            iiif_image_url=iiif_image_url,
            source_url=source_url,
            richelieu_url=richelieu_url,
            date_lower=date_lower,
            date_upper=date_upper,
            institution=institution
        )

        # 2. on ajoute les relations 
        if id_author:
            author = db.session.get(Author, id_author)
            new_icono.author = author
        if id_place:
            # pour new_icono.place et new_icono.theme, on ajoute les relations dans une liste:
            # il s'agit d'une liste de relations
            place = db.session.get(Place, id_place)
            new_icono.place.append(place)
        if id_theme:
            theme = db.session.get(Theme, id_theme)
            new_icono.theme.append(theme)

        # 3. on fait le commit
        # la base de donnée est un système "externe" avec son propre système de validation.
        # un problème est toujours théoriquement possible, donc on met un try...execpt
        try:
            db.session.add(new_icono)
            db.session.commit()

            return True, new_icono
        except Exception as e:
            print(e)
            return False, "Erreur à l'insertion dans la base de données."
```

**Remarques**:
- **la fonction `create` reprend exactement** le code présenté plus haut.
- **pas besoin de `app.app_context`**, puisque `create` est définie à l'intérieur d'un modèle SQLAlchemy
- ici, on ne valide pas les données: la vérification sera faite dans le formulaire d'ajout de `Iconography`. Il est par contre capital que la validation de données ait lieu.

**`create` est une `staticmethod` de la classe `Iconography`**:
- une `staticmethod` est une méthode de `Iconography` qui n'hérite pas d'une instance de la classe (`self`). Elle n'agit donc pas sur un objet `Iconography`, et **sa présence dans la classe est une question d'organisation de code**.
- pour **utiliser `Iconography.create`**, on fait: `user = Iconography.create(...)`

## 2. Créer des formulaires avec WTForms

Les formulaires, c'est un peu la plaie du développement Web: il faut recevoir des données des utilisateur.ice.s, et donc les valider (par exemple, vérifier que les champs obligatoires sont remplis). En plus de ça, il faut garantir l'accessibilité (et donc s'y connaître en accessibilité HTML). Ça peut vite devenir complexe et douloureux à gérer, et c'est répétitif.

On peut créer nos formulaires à la main (en faisant des `<form>` HTML et gérant la validation à la main). Cependant, **des librairies implémentent déjà toute la logique dont on a besoin pour créer des formulaires**.

On va donc apprendre à utiliser **[WTForms](https://wtforms.readthedocs.io/en/3.2.x/), et [Flask-WTF](https://flask-wtf.readthedocs.io/en/1.2.x/)** (plugin pour intégrer WTForms à Flask).

Avec WTForms, on a besoin de 3 choses: 
- **une classe Python** qui définit le formulaire
- **une template HTML** qui sera l'interface utilisateur pour le formulaire
- **une route** qui permette d'accéder au formulaire et de soumettre le formulaire.

### Définir les valeurs possibles pour les relations

Plus haut, on a définit les relations (`Iconography.place`, `Iconography.theme`, `Iconography.author`) en sélectionnant des lignes des autres tables via leurs IDs. 

Avant de créer le formulaire, **pour être plus user-friendly, on va générer une liste de `place`, `iconography` et `theme`** à partir desquelles on pourra sélectionner dans notre formulaire les valeurs de `Iconography.place`, `Iconography.theme`, `Iconography.author`.

Pour ça, on crée dans [`forms.py`](./apps/s5/create_iconography/app/models/forms.py) une fonction `get_iconography_relationships()`. **Cette fonction ne fait que combiner des choses qu'on a déjà vues**:

```py
def get_iconography_relationships() -> Tuple[List, List, List]:
    """
    retourne, pour chaque table avec laquelle Iconography 
    a une relation, une liste de (id, nom_de_objet) (chaque 
    item de la liste est une instance de la table).
    ces listes sont utilisées pour créer les champs des formulaires
    Iconography qui portent sur d'autres tables. 

    NOTE: en HTML, les valeurs sont représentées par des strings. 
    donc, on convertit tous nos ids en strings.
    """
    with app.app_context():
        # pour choisir l'auteur associé.e à une ressource 
        # iconographique, on crée une liste de `(id_auteur, nom_auteur)` 
        all_authors = db.session.execute(
            db.select(Author).order_by(Author.author_name)
        ).scalars().all()
        author_choices = [("", "Choisir un.e auteur.ice") ]
        for author in all_authors:
            author_choices.append(( str(author.id), author.author_name ))

        # pareil pour les thèmes
        all_themes = db.session.execute(
            db.select(Theme).order_by(Theme.theme_name)
        ).scalars().all()
        theme_choices = [ ("", "Choisir un thème") ]
        for theme in all_themes:
            theme_choices.append(( str(theme.id), theme.theme_name ))

        # pareil pour les lieux
        all_places = db.session.execute(
            db.select(Place).order_by(Place.address)
        ).scalars().all()
        place_choices = [ ("", "Choisir un lieu") ]
        for place in all_places:
            place_choices.append(( str(place.id), place.address ))
    return author_choices, theme_choices, place_choices
```

### La classe `IconographyCreateForm`

On l'a dit, [`app/models/forms.py`](`./apps/s5/create_iconography/app/models/forms.py`) stockera toutes nos classes WTForms.

**Notre formulaire aura un champ par colonne ou relation du modèle `Iconography`**.

**Voici `IconographyCreateForm`**, qui permet de créer un formulaire.

```py
from flask_wtf import FlaskForm
from wtforms import StringField, URLField, IntegerField, SelectField
from wtforms.validators import DataRequired, Length, Optional

class IconographyCreateForm(FlaskForm):
    # on récupère les choix pour les tables de relations
    author_choices, theme_choices, place_choices = get_iconography_relationships()

    # on définit un champ par colonne de la table Iconography
    title = StringField(
        "Titre de la ressource", validators=[DataRequired(), Length(max=50)]
    )
    iiif_manifest_url = URLField(
        "URL du manifeste IIIF", validators=[DataRequired()]
    )
    iiif_image_url = URLField(
        "URL de l'image principale", validators=[DataRequired()]
    )
    source_url =  URLField(
        "URL de la source", validators=[Optional()]
    )
    richelieu_url = URLField(
        "URL sur le site Quartier Richelieu", validators=[DataRequired()]
    )
    date_lower = IntegerField(
        "Date", validators=[Optional()]
    )
    date_upper = IntegerField(
        "Date de fin (si nécessaire)", validators=[Optional()]
    )
    institution = StringField(
        "Institution", validators=[DataRequired()]
    )
    
    # on créée des champs pour les jointures
    id_author = SelectField(
        "Auteur", choices=author_choices, validators=[Optional()]
    )
    id_place = SelectField(
        "Lieu", choices=place_choices, validators=[Optional()]
    )
    id_theme = SelectField(
        "Thème", choices=theme_choices, validators=[Optional()]
    )
```

C'est assez simple:
- **notre formulaire est représenté par une classe Python qui hérite de `FlaskForm`**: `class IconographyCreateForm(FlaskForm)`
- **chaque champ du formulaire que l'utilisateur.ice pourra remplir est une propriété de la classe** `IconographyCreateForm`
- **chaque propriété de `Iconography` est associée à un champ du formulaire** 
- **chaque champ est associé à un `Field`**: `StringField`, `UrlField`... Le `Field`, c'est le type de champ. WTForms en définit [un certain nombre](https://wtforms.readthedocs.io/en/3.2.x/fields/#basic-fields). Chaque `Field` prend plusieurs arguments:
    - **en 1er, le nom du champ** qui sera affiché à l'utilisateur
    - **l'argument `validators`** définit une liste avec toutes les règles de validation qui seront associées à ce champ. Il y a [beaucoup de validateurs possibles](https://wtforms.readthedocs.io/en/3.2.x/validators/).
- **pour les relations, on crée un `SelectField`**. Il prend un argument `choices` auquel on passe une liste de `(id, valeur_a_afficher)`: c'est la liste de toutes les options parmi lesquelles l'utilisateur.ice pourra choisir. 

**La syntaxe est donc**:
```py
class MonFormulaire(FlaskForm):
    # 1 champ par propriété de notre classe `Iconography`
    champ = StringField(
        # le titre affiché côté utilisateur.ice
        "Label du champ", 
        # une liste contenant toutes les contraintes que l'utilisateur.ice devra respecter. 
        # ces contraintes doivent correspondes à celles définies dans notre classe `Iconography`
        validators=[...]
    )
```

Tous les formulaires qu'on créera suivront la même logique.

## 3. La template `icono_create.html`

La template HTML [`app/templates/pages/icono_create.html`](./apps/s5/create_iconography/app/templates/pages/icono_create.html) permet de **créer une interface utilisateur pour le formulaire**, via une template HTML. Voici son contenu:

```html
{% extends "base.html" %}

{% block title_extra %}| Créer une ressource iconographique{% endblock %}

{% block main_content %}
    <h1 class="title">Créer une ressource iconographique</h1>

    <!-- 
        method="POST": l'envoi du formulaire correspond à une requête HTTP POST
        action="{{ url_for('icono_create') }}": on ira taper sur la route `icono_create` quand on enverra les données du formulaire
     -->
    <form method="POST" action="{{ url_for('icono_create') }}">
        <!-- obligatoire pour qu'un champ soit valide, permet de garantir l'intégrité des données -->
        {{ form.csrf_token }} 
        <dl>
            <!-- on intère sur toutes les propriétés de notre classe `IconographyCreateForm`: 
                title, date_lower, institution... -->
            {% for field in form %}
                <!-- CSRFTokenField est le form.csrf_token, déjà présent au dessus => on le masque ici -->
                {% if field.type != 'CSRFTokenField' %}
                    <!-- le titre du champ. field.label, c'est le 1er argument de nos `Fields` définis dans `IconographyCreateForm` -->
                    <dt>
                        {{ field.label }}
                        {% if field.flags.required %}*{% endif %}
                    </dt>
                    <dd>                                        
                        <!-- l'input utilisateur, c'est {{ field }}. le HTML valide est généré par 
                         WTForms en fonction du type de `Field`. -->
                        {{ field }}
                        <!-- si il y a des erreurs pour un champ, on affiche l'erreur en dessous de ce champ 
                            (p.ex: une valeur n'est pas un entier) -->
                        {% if field.errors %}
                            <ul class="errors">
                                {% for error in field.errors %}
                                    <li>{{ error }}</li>
                                {% endfor %}
                            </ul>
                        {% endif %}
                    </dd>
                {% endif %}
            {% endfor %}
        </dl>
        <!-- input type="submit", c'est le bouton qui permet d'envoyer un formulaire -->
        <input type="submit" value="Créer" class="button is-rounded negative">
    </form>
{% endblock %}
```

**En résumé**:
- **on crée un `<form>`** qui définit la méthode HTTP et la route associée au formulaire 
- **on intère sur tous les champs** définis dans notre classe `IconographyCreateForm`
- **on finit par un `<input>`** qui permet de confirmer l'envoi du formulaire.
- tout le reste est géré par WTForms.

À noter: l'utilisation de `<form>` et de `<input>` n'est pas propre à WTForms et se retrouve dans tous les formulaires.

## 4. La route `icono_create`

On a maintenant une classe formulaire et une template pour ce formulaire.

Maintenant, on va **créer la route `icono_create` pour accéder au formulaire et pouvoir le soumettre**. Cette route se trouve dans [`app/routes/generic.py`](./apps/s5/create_iconography/app/routes/generic.py)

```py
@app.route("/iconographie/nouveau", methods=["GET", "POST"])
def icono_create():
    """
    vue pour créer une nouvelle ressource iconographique
    """
    form = IconographyCreateForm()

    if form.validate_on_submit():
        title = form.title.data
        iiif_manifest_url = form.iiif_manifest_url.data
        iiif_image_url = form.iiif_image_url.data
        source_url = form.source_url.data
        richelieu_url = form.richelieu_url.data
        date_lower = form.date_lower.data
        date_upper = form.date_upper.data
        institution = form.institution.data

        # on rétroconvertit en int les IDs qui sont représentés par des strings côté formulaire.
        id_author = None
        if form.id_author.data:
            id_author = int(form.id_author.data)
        id_theme = None
        if form.id_theme.data:
            id_theme = int(form.id_theme.data)
        id_place = None
        if form.id_place.data:
            id_place = int(form.id_place.data)

        success, data = Iconography.create(
            title=title,
            iiif_manifest_url=iiif_manifest_url,
            iiif_image_url=iiif_image_url,
            source_url=source_url,
            richelieu_url=richelieu_url,
            date_lower=date_lower,
            date_upper=date_upper,
            institution=institution,
            id_author=id_author,
            id_place=id_place,
            id_theme=id_theme
        )
        if success: 
            flash(f"Nouvelle ressource iconographique créée avec succès: {data.id}", "success")
            return redirect(url_for("icono_main", id_icono=data.id))
        else:
            flash(f"Erreur à la création d'une ressource iconographique: {data}", "error")
            return render_template("pages/icono_create.html", app_name=APP_NAME, form=form)
    
    return render_template("pages/icono_create.html", app_name=APP_NAME, form=form)
```

**Explication de code**:
- **notre route accepte 2 méthodes HTTP**: cela est défini dans le `@app.route()` avec `methods=["GET", "POST"]`
    - `GET` est utilisé pour **accéder au formulaire** sans soumettre de données
    - `POST` est utilisé pour **soumettre le formulaire** (rappelez vous de la template, où on voit `<form method="POST">`)
- **la route a 2 branches correspondantes**
    - `if form.validate_on_submit()` confirme que on a envoyé une requête `POST` et que le formulaire est valide **=> on tente une insertion** avec `Iconography.create()`
    - sinon, (c'est une requête `GET` pour accéder au formulaire ou les données ne sont pas valides), **on renvoie le formulaire** `icono_create.html`.
- **`flash` est utilisé pour afficher les messages de succès ou d'erreur**. flash est une fonction Flask qui prend en 1er argument les messages à afficher, en 2e argument le statut du message (`success` ou `error`).

Pour afficher les messages flashés, **on ajoute à [`base.html`](./apps/s5/create_iconography/app/templates/base.html) le bloc suivant**:

```html
{% with messages = get_flashed_messages(with_categories=True) %}
    {% if messages %}
        <div class="flex-center">
            <ul class=flashes>
                {% for category, message in messages %}
                    <li class=
                        {% if category=="error" %}
                            "flash-message errors"
                        {% else %}
                            "flash-message success"
                        {% endif %}
                    >{{ message }}</li>
                {% endfor %}
            </ul>
        </div>
    {% endif %}
{% endwith %}
```

## En résumé

**Créer un formulaire, c'est**:
- créer une classe `WTForms`
- créer une template HTML pour ce formulaire
- créer une route qui permette d'accéder au formulaire avec `GET` et de soumettre les résultats avec `POST`.

> **Lancer l'application `apps/s5/create_iconography`**:
> ```py
> python ./apps/s5/create_iconography/main.py
> ```

**Essayons de**:
- créer une nouvelle ressource iconographique
- fournir des mauvaises données au formulaire pour voir comment elles sont gérées (mauvais format d'URL, champs requis manquants...)

**En résumé, voici le processus pour insérer des données**:

![pipeline](./img/icono_create_pipeline.png)


---

# `UPDATE`: modifier un objet `Iconography`

Créer des ressources iconographiques nous a demandé de prendre en main:
- les inserts SQLAlchemy pour ajouter de nouveaux objets `Iconography`, et leurs jointures
- les formulaires WTForms
- les routes `POST` en Flask
- et surtout, de connecter tout ça dans le "workflow" le plus complexe qu'on ait fait.

La bonne nouvelle, c'est que **les *updates* fonctionnent quasiment comme des *create***. 

Pour rappel, une update permet de mettre à jour un ou plusieurs objets existants dans la base et **correspond à une requête SQL `UPDATE`**.

**Avant de voir comment faire des updates dans l'application, voyons comment marchent les updates en SQLAlchemy**.

**Notre scénario**, c'est de pouvoir mettre à jour 1 objet `Iconography` et pouvoir changer tous ses champs, ainsi que ses relations. **La pipeline d'upodate correspond donc à**:
- sélectionner un objet `Iconography` par son ID
- lui appliquer des modifications
- sauvegarder ces modifications.

On a vu que:
- chaque ligne de notre table est représentée par une instance de la classe `Iconography`
- chaque colonne d'une table est représentée par un attribut de la classe 
- pour créer une nouvelle ressource iconographique, il suffit de créer un objet avec `Iconography(<valeurs>)`.

Logiquement, **pour modifier un objet `Iconography`, il faut sélectionner l'objet, puis modifier ses attributs**.

## 1. Sélectionner un objet `Iconography` à modifier

On modifie l'objet crée plus tôt dans ce cours.

In [75]:
print(new_icono_id)

with app.app_context():
    # on sélectionne l'objet à modifier et on l'assigne à `icono_item`
    icono_item = db.session.get(Iconography, new_icono_id)
    
    print("icono_item:", icono_item)
    print("title:", icono_item.title)
    print("author:", icono_item.author)
    print("nom de l'auteur:", icono_item.author)
    print("theme:", icono_item.theme)    
    print("nom du theme:", icono_item.theme[0].theme_name)  # pour rappel, Iconography <-> Theme est une relation many-to-many, donc Iconography.theme stocke une liste d'objets Theme 

1312
icono_item: <Iconography 1312>
title: Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]
author: <Author 235>
nom de l'auteur: <Author 235>
theme: [<Theme 13>]
nom du theme: affiche



## 2. Modifier une `Iconography` côté Python

On modifie le titre:

In [76]:
icono_item.title = "tartempion"

print("nouveau titre:", icono_item.title)

nouveau titre: tartempion


## 3. Modifier une `relationship`

Pour modifier les relations, c'est pareil. Il faut juste:
- **sélectionner l'objet qu'on veut associer** à notre `Iconography` (sélectionner un.e auteur.ice ou un thème, par exemple)
- **l'assigner au champ `relationship`**:

In [81]:
# on va modifier l'auteur et ajouter un thème 
with app.app_context():
    # 1. on sélectionne l'auteur.ice
    new_author = db.session.execute(
        db.select(Author).filter(Author.author_name=="Adam, Victor (1801-1866). Lithographe")
    ).scalars().one()
    print("nouvel auteur: ", new_author)
    print("nom du nouvel auteur: ", new_author.author_name)

    # 2. on sélectionne le thème
    new_theme = db.session.execute(
        db.select(Theme).filter(Theme.theme_name=="la commune de paris")
    ).scalars().one()
    print("nouveau thème: ", new_theme)
    print("nom du nouveau thème: ", new_theme.theme_name)

    # 3. on assigne
    icono_item.author = new_author
    icono_item.theme.append(new_theme)  # on ajoute le nouveau thème à la liste des thèmes

    print("auteur mis à jour:", icono_item.author, icono_item.author.author_name)
    print("thèmes mis à jour:", icono_item.theme)
    for theme in icono_item.theme:
        print(f"> titre du thème {theme.id}: {theme.theme_name}")


nouvel auteur:  <Author 161>
nom du nouvel auteur:  Adam, Victor (1801-1866). Lithographe
nouveau thème:  <Theme 36>
nom du nouveau thème:  la commune de paris
auteur mis à jour: <Author 161> Adam, Victor (1801-1866). Lithographe
thèmes mis à jour: [<Theme 13>, <Theme 36>, <Theme 36>]
> titre du thème 13: affiche
> titre du thème 36: la commune de paris
> titre du thème 36: la commune de paris


## 4. Sauvegarder les modifications

Comme on l'a fait en créant des `Iconography`, il suffit **d'ajouter l'objet `Iconography` modifié à notre `session` et de `commit` nos changements**:

In [82]:
with app.app_context():
    try:
        db.session.add(icono_item)
        db.session.commit()
    except Exception as e:
        print("Erreur à l'insertion dans la base de données !", e)

    print("après sauvegarde:", icono_item.title)
    print("après sauvegarde:", icono_item.author)
    print("après sauvegarde:", icono_item.theme)
    

Erreur à l'insertion dans la base de données ! Can't attach instance <Author at 0x713ac1cef440>; another instance with key (<class '__main__.Author'>, (161,), None) is already present in this session.
après sauvegarde: tartempion
après sauvegarde: <Author 161>
après sauvegarde: [<Theme 13>, <Theme 36>, <Theme 36>]



---

# Mettre à jour une ressource iconographique dans l'application

De la même manière que le code SQLAlchemy pour faire un *update* est très semblable au code pour faire des *create*, **le code pour faire des *updates* dans l'application est très très semblable à celui pour faire des *create***. 

## Organisation du code

```txt
└── app
    ├── models
    │   ├── forms.py    # on transforme `IconographyCreateForm` en `IconographyCreateOrUpdateForm`
    │   └── data.py     # on ajoute `Iconography.update()`
    ├── routes
    │   └── generic.py  # on ajoute ici une route pour mettre à jour une ressource 
    └── templates
        ├── pages
        │   └── icono_update.html  # template HTML pour mettre à jour une ressource icono.
        └── includes   
            └── form_fields.html   # tous nos formulaires transforment des classes WTForms en HTML de la même manière. On crée donc un `includes` qui représente le HTML de l'intérieur d'un formulaire. 
```

## 1. Ajouter `Iconography.update()`

Comme pour la création de nouvelles ressources iconographiques, **on crée une méthode `Iconography.update()`** qui 
- **prenne en paramètres** toutes nouvelles valeurs pour l'objet `Iconography`
- **applique les modifications**
- **sauvegarde les changements** en base.

On voit ces changements dans [`update_iconography/app/models/data.py`](./apps/s5/update_iconography/app/models/data.py):

```py
class Iconography(db.Model):
    # j'omets ici le reste de la définition de la classe

    def update(
        self,
        title: str,
        iiif_manifest_url: str,
        iiif_image_url: str,
        source_url: Optional[str],
        richelieu_url: str,
        date_lower: Optional[int],
        date_upper: Optional[int],
        institution: str,
        id_author: Optional[int],
        id_place: Optional[int],
        id_theme: Optional[int],
    ) -> Tuple[bool, Union["Iconography",str]]:
        """
        modifier un objet icono existant en base.
        :returns: (bool, icono|str).
            - bool indique le succès de l'update (si True, l'update a réussi)
            - si l'update a réussi, on retourne l'objet mis à jour, sinon on retourne un message d'erreur.
        """        
        # 1. on met à jour tous les champs
        # qu'est-ce que représente ce `self` ?? 
        self.title = title
        self.iiif_manifest_url = iiif_manifest_url
        self.iiif_image_url = iiif_image_url
        self.source_url = source_url
        self.richelieu_url = richelieu_url
        self.date_lower = date_lower
        self.date_upper = date_upper
        self.institution = institution

        # 2. on ajoute les relations 
        # on remplace les anciennes relations par des nouvelles, donc on 
        # n'utilise pas `append` ici, contrairement à sans `Iconography.create()`
        if id_author:
            author = db.session.get(Author, id_author)
            self.author = author
        else:
            self.author = None
        if id_place:
            place = db.session.get(Place, id_place)
            self.place = [place]
        else:
            self.place = []
        if id_theme:
            theme = db.session.get(Theme, id_theme)
            self.theme = [theme]
        else:
            self.theme = []

        # 3. on sauvegarde les changements
        try:
            db.session.add(self)
            db.session.commit()
            return True, self
        except Exception as e:
            db.session.rollback()
            print(e)
            return False, "Erreur dans la mise à jour"
```

Presque le seul changement avec `Iconography.create()`, c'est que **`update()` n'est pas une `staticmethod` et prend un `self` en 1er attribut**.

En fait, **`self` représente l'instance d'`Iconography` que l'on veut modifier**. On utilise donc `update()` sur une instance d'`Iconography`:

```py
# on sélectionne l'item à modifier
icono_item = db.session.get(Iconography, 41)
# l'objet `icono_item` devient le `self` utilisé dans `Iconography.update()`
icono_item.update(title="tartempion", <autres valeurs>)
```

À noter, ce type de définition de méthode avec un `self` est le "cas normal" pour la création de méthodes de classes, et les `staticmethod`s sont plutôt des cas particuliers: on utilise généralement des méthodes pour modifier un objet.

## 2. Formulaires WTForms: transformer `IconographyCreateForm` en `IconographyCreateOrUpdateForm`

On l'a vu, **un *update*, c'est comme un *create*, sauf qu'on modifie les valeurs d'un objet préexistant** au lieu de créer un nouvel objet.

Du point de vue d'un formulaire, cela veut dire que le formulaire d'*update* est le même que pour les *create*, sauf qu'on donne **comme valeurs par défaut les valeurs de l'objet qu'on vient modifier**:

![form_create_or_update](./img/form_create_or_update.png)

> Pour voir le résultat "en vrai", lancer l'appli `update_iconography`
> ```py
> python apps/s5/update_iconography/main.py
> ```
>
> Et comparer: 
> - [`http://localhost:5000/iconographie/nouveau`](http://localhost:5000/iconographie/nouveau)
> - [`http://localhost:5000/iconographie/1/modifier`](http://localhost:5000/iconographie/1/modifier)

```py
class IconographyCreateOrUpdateForm(FlaskForm):
    # on récupère les choix pour les tables de relations
    # author_choices, theme_choices, place_choices = get_iconography_relationships()

    # on récupère les choix pour les tables de relations
    # à noter, les ID retournés par `get_iconography_relationships` sont représentés par des `str`.
    # il faudra les rétroconvertir à l'insert ou update
    author_choices, theme_choices, place_choices = get_iconography_relationships()

    # on définit un champ par colonne de la table Iconography
    title = StringField(
        "Titre de la ressource", validators=[DataRequired()]
    )
    iiif_manifest_url = URLField(
        "URL du manifeste IIIF", validators=[DataRequired()]
    )
    iiif_image_url = URLField(
        "URL de l'image principale", validators=[DataRequired()]
    )
    source_url =  URLField(
        "URL de la source", validators=[Optional()]
    )
    richelieu_url = URLField(
        "URL sur le site Quartier Richelieu", validators=[DataRequired()]
    )
    date_lower = IntegerField(
        "Date", validators=[Optional()]
    )
    date_upper = IntegerField(
        "Date de fin (si nécessaire)", validators=[Optional()]
    )
    institution = StringField(
        "Institution", validators=[DataRequired()]
    )
    
    # on créée des champs pour les jointures
    id_author = SelectField(
        "Auteur", choices=author_choices, validators=[Optional()]
    )
    id_place = SelectField(
        "Lieu", choices=place_choices, validators=[Optional()]
    )
    id_theme = SelectField(
        "Thème", choices=theme_choices, validators=[Optional()]
    )

    def __init__(self, icono_item: Iconography = None, *args, **kwargs):
        super().__init__(*args, **kwargs)
                
        # on définit les défauts seulement si `icono_item` est défini
        if icono_item and not self.is_submitted():
            
            # on récupère les ID de chaque relation à `icono_item`. ces ID seront 
            # utilisés comme défaut de `id_author`, `id_theme` et `id_place`
            id_author_default = None
            if icono_item.author:
                # on convertit nos ID par défaut en str pour coller avec ce qui est retourné par `get_iconography_relationships` 
                id_author_default = str(icono_item.author.id) 
            id_place_default = None 
            if len(icono_item.place):
                id_place_default = str(icono_item.place[0].id)
            id_theme_default = None 
            if len(icono_item.theme):
                id_theme_default = str(icono_item.theme[0].id)

            # on utilise `icono_item` pour définir toutes les valeurs par défaut du formulaire
            self.title.data = icono_item.title
            self.iiif_manifest_url.data = icono_item.iiif_manifest_url
            self.iiif_image_url.data = icono_item.iiif_image_url
            self.source_url.data = icono_item.source_url
            self.richelieu_url.data = icono_item.richelieu_url
            self.date_lower.data = icono_item.date_lower
            self.date_upper.data = icono_item.date_upper
            self.institution.data = icono_item.institution

            self.id_author.data = id_author_default
            self.id_place.data = id_place_default
            self.id_theme.data = id_theme_default
```

### 3. Templates HTML

#### Un composant pour représenter le contenu d'un formulaire: `includes/form_fields.html`

```html
{{ form.csrf_token }} 
<dl>
    {% for field in form %}
        {% if field.type != 'CSRFTokenField' %}
            <dt>
                {{ field.label }}
                {% if field.flags.required %}*{% endif %}
            </dt>
            <dd>
                {{ field }}
                {% if field.errors %}
                    <ul class="errors">
                        {% for error in field.errors %}
                            <li>{{ error }}</li>
                        {% endfor %}
                    </ul>
                {% endif %}
            </dd>
        {% endif %}
    {% endfor %}
</dl>
<input type="submit" value="{{ submit_label }}" class="button is-rounded negative">
```

#### Utilisation de `form_fields.html`: `icono_update.html`

```html
{% extends "base.html" %}

{% block title_extra %}| Mettre à jour une ressource iconographique{% endblock %}

{% block main_content %}
    <h1 class="title">Mettre à jour une ressource iconographique</h1>

    <form method="POST" action="{{ url_for('icono_update', id_icono=icono_item.id) }}">
        {% with form=form, submit_label="Enregistrer" %} 
            {% include "includes/form_fields.html" %}
        {% endwith %}
    </form>
{% endblock %}
```

### 4. Une route pour les mises à jour: `icono_update` 

```py
@app.route("/iconographie/<int:id_icono>/modifier", methods=["GET", "POST"])
def icono_update(id_icono: int):
    """
    vue pour modifier une ressource iconographique existante
    """
    # todo Iconography.update() method 

    icono_item = db.get_or_404(Iconography, id_icono)
 
    form = IconographyCreateOrUpdateForm(icono_item=icono_item)
 
    if form.validate_on_submit():
        title = form.title.data
        iiif_manifest_url = form.iiif_manifest_url.data
        iiif_image_url = form.iiif_image_url.data
        source_url = form.source_url.data
        richelieu_url = form.richelieu_url.data
        date_lower = form.date_lower.data
        date_upper = form.date_upper.data
        institution = form.institution.data

        # on rétroconvertit en int les IDs qui sont représentés par des strings côté formulaire.
        id_author = None
        if form.id_author.data:
            id_author = int(form.id_author.data)
        id_theme = None
        if form.id_theme.data:
            id_theme = int(form.id_theme.data)
        id_place = None
        if form.id_place.data:
            id_place = int(form.id_place.data)
            
        success, data = icono_item.update(
            title=title,
            iiif_manifest_url=iiif_manifest_url,
            iiif_image_url=iiif_image_url,
            source_url=source_url,
            richelieu_url=richelieu_url,
            date_lower=date_lower,
            date_upper=date_upper,
            institution=institution,
            id_author=id_author,
            id_place=id_place,
            id_theme=id_theme
        )
        if success: 
            flash(f"Ressource iconographique mise à jour avec succès: {data.id}", "success")
            return redirect(url_for("icono_main", id_icono=data.id))
        else:
            flash(f"Erreur à la mise à jour d'une ressource iconographique: {data}", "error")
            return render_template("pages/icono_update.html", app_name=APP_NAME, form=form, icono_item=icono_item)
    
    return render_template("pages/icono_update.html", app_name=APP_NAME, form=form, icono_item=icono_item)
```




- proof of concept dans le notebook
- dans l'appli:
    - templates: `form_fields.html` + `icono_update.html`
    - forms: un update, côté formulaire, c'est comme un create sauf qu'on fournit comme valeurs par défaut les valeurs de l'objet qu'on modifie => on transforme `IconographyCreateForm` en `IconographyCreateOrUpdateForm`, qui
        - si un objet `icono_item` est passé, utilise ses valeurs comme défaut => scénario update
        - si aucun objet n'est passé, pas de défauts => scénario create
    - route: à 99% la même que icono_create
    - Iconography.update: same, c'est quasi la même. 

- update
- delete